# Projekt MSP1 / 2025
Cílem tohoto projektu je se seznámit s programovými nástroji využívaných ve statistice a osvojit si základní procedury. Projekt není primárně zaměřen na efektivitu využívání programového vybavení (i když úplně nevhodné konstrukce mohou mít vliv na hodnocení), ale nejvíce nás zajímají vaše statistické závěry a způsob vyhodnocení. Dbejte také na to, že každý graf musí splňovat nějaké podmínky - přehlednost, čitelnost, popisky.

V projektu budete analyzovat časy běhu šesti různých konfigurací algoritmů. Ke každé konfiguraci vzniklo celkem 500 nezávislých běhů, jejichž logy máte k dispozici v souboru [logfiles.zip](logfiles.zip).

Pokud nemáte rozchozené prostředí pro pro spouštění Jupyter notebooku, můžete využit službu [Google Colab](https://colab.google/). Jakákoliv spolupráce, sdílení řešení a podobně je zakázáno!

S případnými dotazy se obracejte na Davida Hudáka (ihudak@fit.vutbr.cz).

__Odevzdání:__ tento soubor (není potřeba aby obsahoval výstupy skriptů) do pondělí 3. 11. 2025 v IS VUT. Kontrola bude probíhat na Pythonu 3.12.3 (standardní instalace Ubuntu); neočekává se však to, že byste používali nějaké speciality a nekompatibilní knihovny. V případě nesouladu verzí a podobných problémů budete mít možnost reklamace a prokázání správnosti funkce. Bez vyplnění vašich komentářů a závěrů do označených buněk nebude projekt hodnocen!

__Upozornění:__ nepřidávejte do notebooku další buňky, odpovídejte tam, kam se ptáme (textové komentáře do Markdown buněk)

__Tip:__ před odevzdáním resetujte celý notebook a zkuste jej spustit od začátku. Zamezíte tak chybám krokování a editací, kdy výsledek z buňky na konci použijete na začátku.

__OTÁZKA K DOPLNĚNÍ:__

_Jméno a login autora_

David Kvaček (xkvace00@stud.fit.vutbr.cz)

## Načtení potřebných knihoven
Načtěte knihovny, které jsou nutné pro zpracování souborů a práci se statistickými funkcemi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
import json
import re
import math
from zipfile import ZipFile

## Načtení dat do DataFrame
Ze souboru `logfiles.zip` umístěném ve stejném adresáři načtěte data a vytvořte Pandas DataFrame.

Výsledky jsou uložené ve formátu JSON - pro zpracování použijte knihovnu `json`.
Můžete využít následující kostru - je vhodné pracovat přímo se ZIP souborem. Jedinou nevýhodou může být to, že vám bude vracet _byte_ objekt, který musíte přes funkci `decode` zpracovat.

Upravte také pomocí funkce `.astype()` datové typy patřičných sloupců.

```py
data = []
with ZipFile("logfiles.zip") as zf:
    for filename in zf.namelist():
        # TODO test názvu souboru
        with zf.open(filename, "r") as f:
            pass # vytvořte slovník

df = pd.DataFrame(data)
df
```

In [ ]:
data = []
with ZipFile('logfiles.zip', 'r') as zf:
    for filename in zf.namelist():
        if not re.match(r'^log_files/config[1-6]_run[0-4][0-9]{2}\.json$', filename):
            continue
        with zf.open(filename, 'r') as f:
            record = json.loads(f.read().decode('utf-8'))
            data.append(record)

df = pd.DataFrame(data)
df = df.astype({
    'configuration': 'string',
    'run': 'int64',
    'runtime': 'float64',
    'status': 'string'
})
df

## Analýza a čištění dat
Vhodným způsobem pro všechny konfigurace analyzujte časy běhů a pokud tam jsou, identifikujte hodnoty, které jsou chybné. Vyberte vhodný graf, který zobrazí samostatně jednotlivé konfigurace.

In [ ]:
sns.boxplot(
    data=df,
    order=sorted(df['configuration'].unique()),
    x='configuration',
    y='runtime'
)
plt.show()

__OTÁZKA K DOPLNĚNÍ:__

_Objevily se nějaké chybné hodnoty? Proč tam jsou s ohledem na to, že se jedná o běhy algoritmů? Proč jste zvolili tento typ grafu?_

Ano, objevily se chybné hodnoty označené jako `TIMEOUT` a `SEGFAULT` ve sloupci `status`, které značí, že daný běh algoritmu buď překročil časový limit, nebo došlo k chybě při jeho provádění (např. nedostatek paměti).
Pro vizualizaci je zvolen tzv. boxplot graf, který efektivně zobrazuje rozdělení dat, včetně mediánu, kvartilů a potenciálních odlehlých hodnot.
Tento typ grafu je vhodný pro porovnání více konfigurací najednou, což umožňuje snadno identifikovat rozdíly v časech běhů mezi jednotlivými konfiguracemi.

Vyčistěte dataframe `df` tak, aby tam tyto hodnoty nebyly a ukažte znovu analýzu toho, že čištění dat bylo úspěšné. Odtud dále pracujte s vyčištěným datasetem.

In [ ]:
df = df[df['status'] == 'SUCCESS']
sns.boxplot(
    data=df,
    order=sorted(df['configuration'].unique()),
    x='configuration',
    y='runtime'
)
plt.show()

## Deskriptivní popis hodnot
Vypište pro jednotlivé konfigurace základní deskriptivní parametry.  

__TIP__ pokud výsledky uložíte jako Pandas DataFrame, zobrazí se v tabulce.

In [ ]:
df.groupby('configuration')['runtime'].describe()

__OTÁZKA K DOPLNĚNÍ:__

_Okomentujte, co všechno můžeme z parametrů vyčíst._

Z deskriptivních parametrů lze vyčíst:
- __count__: počet úspěšných běhů pro každou konfiguraci, což vypovídá o stabilitě a spolehlivosti dané konfigurace,
- __mean__: průměrný čas běhů, který poskytuje představu o celkové výkonnosti konfigurace,
- __std__: směrodatná odchylka, která ukazuje variabilitu časů běhů; vyšší hodnota naznačuje větší rozptyl v časech,
- __min__ a __max__: minimální a maximální časy běhů, které značí extrémní hodnoty a mohou indikovat potenciální problémy nebo výjimečné případy,
- __25 %__, __50 % (medián)__ a __75 %__: kvartily, které poskytují informace o rozdělení časů běhů a pomáhají identifikovat typické hodnoty a odlehlé případy.

## Vizualizace
Vizualizujte časy běhů algoritmů tak, aby byl v jednom grafu zřejmý i rozptyl hodnot, avšak bylo možné porovnání. Zvolte vhodný graf, který pak níže komentujte.

In [ ]:
sns.violinplot(
    data=df,
    order=sorted(df['configuration'].unique()),
    x='configuration',
    y='runtime'
)
plt.show()

__OTÁZKA K DOPLNĚNÍ:__

_Okomentujte výsledky z tabulky._

Z grafu a výše uvedené tabulky s deskriptivními parametry je patrné, že konfigurace č. 1 nemá normální rozdělení časů běhů, což může být způsobeno přítomností odlehlých hodnot nebo nerovnoměrným rozdělením dat.
Konfigurace č. 2, 3 a 5 vykazují relativně nízké směrodatné odchylky, což naznačuje konzistentní výkonnost, avšak jejich mediány a průměrné časy běhů jsou zřetelně vyšší než u zbývajících konfigurací.
Konfigurace č. 4 a 6 mají tyto hodnoty naopak výrazně nižší, přičemž si zachovávají poměrně akceptovatelné směrodatné odchylky.

## Určení efektivity konfigurací algoritmů
Nás ale zajímá, jaká konfigurace je nejrychlejší. Z výše vykresleného grafu můžeme vyloučit některé konfigurace. Existuje tam však minimálně jedna dvojice, u které nedokážeme jednoznačně určit, která je lepší - pokud nebudeme porovnávat pouze extrémní hodnoty, které mohou být dané náhodou, ale celkově. Proto proveďte vhodný test významnosti - v následující části diskutujte zejména rozložení dat (i s odkazem na předchozí buňky, variabilitu vs polohu a podobně). Je nutné každý logický krok a výběry statistických funkcí komentovat. Určete také směr (tzn. která implementace je lepší).

Využijte vhodnou funkci z knihovny `scipy.stats` a funkci poté __implementujte sami__ na základě základních matematických funkcí knihovny `numpy` případně i funkcí pro výpočet vhodného rozložení v [scipy.stats](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.html). Při vlastní implementaci není nutné se primárně soustředit na efektivitu výpočtu (není potřeba využít všechny funkce numpy, můžete použít normální cykly a podobně - v hodnocení však bude zahrnuta přehlednost a neměly by se objevit jasné chyby, jako je zvýšení třídy složitosti a podobně). V případě, že pro řešení úlohy využijete více testů, stačí implementovat pouze jeden. 

__OTÁZKA K DOPLNĚNÍ:__

_Jaká data budete zkoumat? Jaké mají rozložení a parametry (např. varianci) a jaký test použijete? Jaká je nulová hypotéza? Jak se liší variabilita a poloha vybraných konfigurací?_

Ze zadání plyne, že zkoumanými daty budou časy běhů algoritmů `runtime` vybrané dvojice konfigurací, které vykazují podobné výkonnostní charakteristiky (v tomto případě konfigurace č. 4 a 6).
Zvolená data mají na základě předchozích analýz a vizualizací přibližně normální rozdělení s následujícími parametry:

- konfigurace č. 4:
    - __mean__ (průměrný čas běhu): `120.54`,
    - __std__ (směrodatná odchylka): `48.42`,
    - __var__ (variance/rozptyl): `2,344.59`,
    - __min__ (minimální čas běhu): `6.48`,
    - __50 %__ (medián): `117.86`,
    - __max__ (maximální čas běhu): `269.26`,

- konfigurace č. 6:
    - __mean__ (průměrný čas běhu): `112.67`,
    - __std__ (směrodatná odchylka): `39.12`,
    - __var__ (variance/rozptyl): `1,530.26`,
    - __min__ (minimální čas běhu): `1.87`,
    - __50 %__ (medián): `111.74`,
    - __max__ (maximální čas běhu): `237.72`.

Pro porovnání těchto dvou konfigurací je zvolen neparametrický Mann–Whitney U test, který je vhodný pro porovnání dvou nezávislých výběrů.
Nulová hypotéza H<sub>0</sub> tohoto testu je, že distribuce časů běhů konfigurace č. 6 je nižší než distribuce časů běhů konfigurace č. 4 (zvolena v závislosti na parametrech rozložení výše), což by naznačovalo, že právě konfigurace č. 6 je efektivnější.
Variabilita a poloha obou konfigurací se liší v tom, že konfigurace č. 6 má nižší průměrný čas běhu, nižší směrodatnou odchylku a celkově jsou její hodnoty časů běhů koncentrovány blíže k průměru, který je rovněž nižší, ve srovnání s konfigurací č. 4.

In [ ]:
config4 = df[df['configuration'] == 'config4']['runtime']
config6 = df[df['configuration'] == 'config6']['runtime']

statistic, pvalue = stats.mannwhitneyu(config6, config4, alternative='less')
print(f'Mann–Whitney U statistic: {statistic}\np–value: {pvalue}')

__OTÁZKA K DOPLNĚNÍ:__

_Jaký je závěr statistického testu?_

Na základě provedeného Mann–Whitney U testu byly získány následující výsledky:

- Mann–Whitney U statistika: `109,347.0`,
- p–hodnota: `7.86e-03`.

Vzhledem k tomu, že p–hodnota je menší než běžně používaná hladina významnosti `0.05`, závěr statistického testu je takový, že s `95 %` věrností lze zamítnout nulovou hypotézu H<sub>0</sub> a přijmout alternativní hypotézu H<sub>1</sub>, že distribuce časů běhů konfigurace č. 6 není skutečně nižší než distribuce časů běhů konfigurace č. 4.

### Vlastní implementace
Implementujte stejný test pomocí knihovních funkcí a ukažte, že je výsledek stejný.

In [ ]:
def my_mannwhitneyu(
        x: np.ndarray,
        y: np.ndarray,
        use_continuity: bool = True,
        alternative: str = 'two-sided'
) -> tuple[float, float]:
    """
    Mann–Whitney U test equivalent of scipy.stats.mannwhitneyu.

    Args:
        x (np.ndarray): Input array of sample x observations.
        y (np.ndarray): Input array of sample y observations.
        use_continuity (bool, optional): Whether to apply continuity correction. Defaults to True.
        alternative (str, optional): Defines the alternative hypothesis. Defaults to 'two-sided'.
            Alternative hypothesis:
            'two-sided': the distributions of both samples are equal.
            'less': the distribution of x is stochastically less than that of y.
            'greater': the distribution of x is stochastically greater than that of y.

    Returns:
        MannWhitneyUResult (tuple[float, float]): U statistic and p–value.
    """

    x = np.asarray(x)
    y = np.asarray(y)
    n1 = len(x)
    n2 = len(y)

    if n1 == 0 or n2 == 0:
        raise ValueError('Input array of sample x and y observations must not be empty.')

    xy = np.sort(np.concatenate([x, y]))
    unique, index, counts = np.unique(xy, return_index=True, return_counts=True)
    ranks = np.zeros_like(xy, dtype=float)

    for i in range(len(unique)):
        ranks[index[i]:index[i] + counts[i]] = 0.5 * (2 * index[i] + counts[i] + 1)

    xranks = ranks[np.isin(xy, x)]
    yranks = ranks[np.isin(xy, y)]
    u1 = np.sum(xranks) - n1 * 0.5 * (n1 + 1)
    u2 = np.sum(yranks) - n2 * 0.5 * (n2 + 1)

    if alternative == 'two-sided':
        u = min(u1, u2)
    elif alternative == 'less':
        u = u1
    elif alternative == 'greater':
        u = u2
    else:
        raise ValueError('Alternative hypothesis must be one of \'two–sided\', \'less\' or \'greater\'.')

    mean = 0.5 * n1 * n2
    std = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12.0)

    if use_continuity:
        correction = 0.5
    else:
        correction = 0.0

    if alternative == 'two-sided':
        if u1 < u2:
            pvalue = 2 * min(
                0.5 * (1 + math.erf(((u1 - mean + correction) / std) / math.sqrt(2))),
                1 - 0.5 * (1 + math.erf(((u1 - mean + correction) / std) / math.sqrt(2)))
            )
        else:
            pvalue = 2 * min(
                0.5 * (1 + math.erf(((u2 - mean - correction) / std) / math.sqrt(2))),
                1 - 0.5 * (1 + math.erf(((u2 - mean - correction) / std) / math.sqrt(2)))
            )

    elif alternative == 'less':
        pvalue = 0.5 * (1 + math.erf(((u - mean + correction) / std) / math.sqrt(2)))
    elif alternative == 'greater':
        pvalue = 1 - 0.5 * (1 + math.erf(((u - mean - correction) / std) / math.sqrt(2)))
    else:
        raise ValueError('Alternative hypothesis must be one of \'two–sided\', \'less\' or \'greater\'.')

    return u, pvalue

statistic, pvalue = my_mannwhitneyu(config6, config4, alternative='less')
print(f'Mann–Whitney U statistic: {statistic}\np–value: {pvalue}')